# Kshetra: field history and model training (free: Google Colab or Kaggle)

1. Connects to your Kshetra database (Supabase free tier or local).
2. Plots one field's NDVI history with detected seasons, crops and stress events.
3. Trains the yield model (XGBoost quantiles) and shows its out-of-year accuracy.

Put your connection string in a Colab **secret** named `DATABASE_URL` (key icon in the left bar). Never paste keys into a cell.

In [ ]:
!git clone -q https://github.com/riyanshareefshaik/Kshetra.git
%cd Kshetra/backend
!pip -q install -r requirements.txt -r ../notebooks/requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata          # Colab secret
    os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
except ImportError:
    pass                                       # Kaggle / local: set DATABASE_URL in the environment

In [ ]:
from app.db.session import get_conn
from app.ml.analyze import load_timeseries

with get_conn() as conn:
    fields = conn.execute("SELECT id::text, name, ingest_status FROM fields ORDER BY name").fetchall()
fields

In [ ]:
import matplotlib.pyplot as plt

FIELD_ID = fields[0]["id"]
with get_conn() as conn:
    ts = load_timeseries(conn, FIELD_ID)
    seasons = conn.execute("SELECT * FROM seasons WHERE field_id = %s ORDER BY sowing_date", (FIELD_ID,)).fetchall()
    events = conn.execute("SELECT * FROM events WHERE field_id = %s ORDER BY start_date", (FIELD_ID,)).fetchall()

fig, ax = plt.subplots(figsize=(14, 4))
obs = ts[ts.ndvi_source == "s2"]
fill = ts[ts.ndvi_source == "sar_fill"]
ax.plot(ts.index, ts.ndvi_smoothed, color="#2f7d32", lw=1.5, label="NDVI (smoothed)")
ax.scatter(obs.index, obs.ndvi, s=6, color="#2f7d32", label="Sentinel-2 (clear)")
ax.scatter(fill.index, fill.ndvi, s=6, color="#8e6bbf", label="Filled from radar")
for s in seasons:
    end = s["harvest_date"] or ts.index[-1]
    ax.axvspan(s["sowing_date"], end, color="#c8e6c9", alpha=0.35)
    label = f"{s['crop']} ({s['crop_confidence']:.0%})"
    if s["yield_mid_t_ha"]:
        label += f"\n{s['yield_low_t_ha']:.1f}-{s['yield_high_t_ha']:.1f} t/ha"
    ax.text(s["peak_date"], 0.95, label, ha="center", va="top", fontsize=8)
for e in events:
    ax.axvline(e["start_date"], color="#c62828", lw=0.8, alpha=0.6)
ax.set_ylim(0, 1); ax.set_ylabel("NDVI"); ax.legend(loc="lower left", fontsize=8)
plt.show()

## Train the yield model

Needs district crop statistics in `district_yields` (see `scripts/load_reference.py`) and at least 30
confident field-seasons. Accuracy is checked leave-one-year-out: `range_coverage` should be near 0.8
(80% of true yields inside the P10-P90 range).

In [ ]:
from scripts.train_models import train_yield
with get_conn() as conn:
    report = train_yield(conn)
report

In [ ]:
!python -m scripts.analyze_all   # refresh every field's yields and reasons with the new model